# AirIntel

# Notebook 03 — Data Cleaning

## Objective

This notebook cleans the validated AirIntel dataset based on the findings from the data validation stage.

Cleaning tasks include:

- Removing unnecessary features
- Handling missing values
- Treating invalid observations
- Standardizing datatypes
- Detecting outliers
- Performing consistency checks

The cleaned dataset will be saved for further enrichment and feature engineering.

In [1]:
# Import required libraries

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# Define project directories

PROJECT_ROOT = Path("..")

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"

In [3]:
# Load dataset

df = pd.read_parquet(
    PROCESSED_DATA / "raw_airintel.parquet"
)

df.shape

(842160, 71)

In [4]:
# Create a copy for cleaning

df_clean = df.copy()

In [5]:
# Sort observations before performing time-series operations

df_clean = (
    df_clean
    .sort_values(["City", "Datetime"])
    .reset_index(drop=True)
)

In [6]:
# Remove columns with 100% missing values

missing_percent = df_clean.isna().mean() * 100

empty_columns = missing_percent[
    missing_percent == 100
].index.tolist()

print(empty_columns)

['Temp_80m_C', 'Temp_120m_C', 'Temp_180m_C', 'Wind_Speed_80m_kmh', 'Wind_Speed_120m_kmh', 'UV_Index', 'NH3_ugm3', 'Inversion_Strength_C']


In [7]:
# Drop empty columns

df_clean.drop(
    columns=empty_columns,
    inplace=True
)

df_clean.shape

(842160, 63)

In [8]:
# Separate numerical features into logical groups

pollutant_columns = [
    "PM2_5_ugm3",
    "PM10_ugm3",
    "NO2_ugm3",
    "SO2_ugm3",
    "CO_ugm3",
    "O3_ugm3"
]

weather_columns = [
    "Temp_2m_C",
    "Humidity_Percent",
    "Dew_Point_C",
    "Wind_Speed_10m_kmh",
    "Pressure_MSL_hPa",
    "Surface_Pressure_hPa",
    "Cloud_Cover_Percent",
    "Rain_mm",
    "Precipitation_mm"
]

In [9]:
# Replace negative pollutant values with missing values

for column in pollutant_columns:

    if column in df_clean.columns:

        df_clean.loc[
            df_clean[column] < 0,
            column
        ] = np.nan

In [10]:
# Display remaining missing values

missing_summary = pd.DataFrame({

    "Missing Values": df_clean.isna().sum(),

    "Missing %": (
        df_clean.isna().mean() * 100
    ).round(2)

})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
]

missing_summary.sort_values(
    "Missing %",
    ascending=False
)

,Missing Values,Missing %
AQI_Category,2516,0.30
US_AQI,145,0.02
US_AQI_PM25,145,0.02
US_AQI_PM10,145,0.02
EU_AQI,145,0.02
EU_AQI_PM25,145,0.02
EU_AQI_PM10,145,0.02
O3_ugm3,82,0.01
US_AQI_O3,73,0.01
NO2_ugm3,2,0.00


In [11]:
# Fill weather observations using forward and backward fill within each city

for column in weather_columns:

    if column in df_clean.columns:

        df_clean[column] = (

            df_clean

            .groupby("City")[column]

            .transform(lambda x: x.ffill().bfill())

        )

In [12]:
# Interpolate pollutant concentrations within each city

for column in pollutant_columns:

    if column in df_clean.columns:

        df_clean[column] = (

            df_clean

            .groupby("City")[column]

            .transform(

                lambda x: x.interpolate(
                    method="linear",
                    limit_direction="both",
                    limit_area="inside"
                )

            )

        )

In [13]:
# Compare missing values before and after cleaning

missing_comparison = pd.DataFrame({

    "Before": df.isna().sum(),

    "After": df_clean.isna().sum()

})

missing_comparison["Difference"] = (
    missing_comparison["After"]
    - missing_comparison["Before"]
)

missing_comparison[
    missing_comparison["Difference"] != 0
].sort_values(
    "Difference",
    ascending=False
)

,Before,After,Difference
Inversion_Strength_C,842160,NaN,NaN
NH3_ugm3,842160,NaN,NaN
Temp_120m_C,842160,NaN,NaN
Temp_180m_C,842160,NaN,NaN
Temp_80m_C,842160,NaN,NaN
UV_Index,842160,NaN,NaN
Wind_Speed_120m_kmh,842160,NaN,NaN
Wind_Speed_80m_kmh,842160,NaN,NaN


In [14]:
# Remove duplicate rows

before = len(df_clean)

df_clean.drop_duplicates(
    inplace=True
)

after = len(df_clean)

print(f"Removed {before-after} duplicate rows.")

Removed 0 duplicate rows.


In [15]:
# Verify duplicate removal

df_clean.duplicated().sum()

np.int64(0)

In [16]:
# Detect potential outliers using the IQR method

iqr_summary = []

for column in pollutant_columns:

    if column in df_clean.columns:

        q1 = df_clean[column].quantile(0.25)

        q3 = df_clean[column].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr

        upper = q3 + 1.5 * iqr

        outliers = (

            (df_clean[column] < lower)

            |

            (df_clean[column] > upper)

        ).sum()

        iqr_summary.append(

            [column, outliers]

        )

iqr_summary = pd.DataFrame(

    iqr_summary,

    columns=[

        "Feature",

        "Potential Outliers"

    ]

)

iqr_summary

,Feature,Potential Outliers
0,PM2_5_ugm3,49144
1,PM10_ugm3,54094
2,NO2_ugm3,67453
3,SO2_ugm3,73210
4,CO_ugm3,65035
5,O3_ugm3,10668


In [17]:
# Convert datetime column

df_clean["Datetime"] = pd.to_datetime(
    df_clean["Datetime"]
)

In [18]:
# Summarize cleaned dataset

clean_summary = pd.DataFrame({

    "Metric":[

        "Rows",

        "Columns",

        "Missing Values",

        "Duplicate Rows"

    ],

    "Value":[

        len(df_clean),

        df_clean.shape[1],

        df_clean.isna().sum().sum(),

        df_clean.duplicated().sum()

    ]

})

clean_summary

,Metric,Value
0,Rows,842160
1,Columns,63
2,Missing Values,3463
3,Duplicate Rows,0


In [19]:
# Save cleaned dataset

df_clean.to_parquet(
    PROCESSED_DATA / "clean_airintel.parquet",
    index=False
)

df_clean.to_csv(
    PROCESSED_DATA / "clean_airintel.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [20]:
# Summarize cleaning results

cleaning_summary = pd.DataFrame({

    "Metric": [

        "Original Rows",
        "Final Rows",
        "Original Columns",
        "Final Columns",
        "Remaining Missing Values",
        "Duplicate Rows",
        "Negative Values Remaining"

    ],

    "Value": [

        len(df),
        len(df_clean),
        df.shape[1],
        df_clean.shape[1],
        df_clean.isna().sum().sum(),
        df_clean.duplicated().sum(),
        (df_clean[pollutant_columns] < 0).sum().sum()

    ]

})

cleaning_summary

,Metric,Value
0,Original Rows,842160
1,Final Rows,842160
2,Original Columns,71
3,Final Columns,63
4,Remaining Missing Values,3463
5,Duplicate Rows,0
6,Negative Values Remaining,0


In [21]:
# Document cleaning decisions

cleaning_decisions = pd.DataFrame({

    "Step": [

        "Empty Columns",
        "Negative Pollutant Values",
        "Weather Missing Values",
        "Pollutant Missing Values",
        "Duplicate Rows",
        "Outliers"

    ],

    "Method": [

        "Dropped",
        "Converted to NaN",
        "Forward/Backward Fill",
        "Linear Interpolation",
        "Removed",
        "Detected (IQR)"

    ],

    "Reason": [

        "No useful information",
        "Physically impossible values",
        "Preserve temporal continuity",
        "Maintain pollution trends",
        "Avoid redundant records",
        "Review before modeling"

    ]

})

cleaning_decisions

,Step,Method,Reason
0,Empty Columns,Dropped,No useful information
1,Negative Pollutant Values,Converted to NaN,Physically impossible values
2,Weather Missing Values,Forward/Backward Fill,Preserve temporal continuity
3,Pollutant Missing Values,Linear Interpolation,Maintain pollution trends
4,Duplicate Rows,Removed,Avoid redundant records
5,Outliers,Detected (IQR),Review before modeling


## Conclusion

The dataset has been cleaned by removing invalid observations, handling duplicate records, treating missing values where scientifically justified, and preserving genuine missing observations to avoid introducing bias.

The cleaned dataset is now ready for external data enrichment.